# 02 Data Cleaning

## Objective

Clean the raw Telco churn dataset so key fields are ready for downstream analysis and modeling. This notebook focuses on the known `TotalCharges` type issue, validates the cleaned result, and saves a cleaned dataset for later stages.


## Imports


In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import load_raw_data
from src.data.data_validation import validate_raw_data


## Load And Validate Raw Data


In [2]:
df = load_raw_data()
validate_raw_data(df)
df_clean = df.copy()

print("Raw dataset loaded and validated successfully.")
print(f"Data source: {project_root / 'data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'}")
print(f"Shape: {df_clean.shape}")


Raw dataset loaded and validated successfully.
Data source: /Users/mohammadmubashir/VCode/Churn/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape: (7043, 21)


## Inspect Cleaning Targets


In [3]:
blank_total_charges = df_clean['TotalCharges'].astype(str).str.strip().eq('')

print(f"TotalCharges dtype before cleaning: {df_clean['TotalCharges'].dtype}")
print(f"Blank TotalCharges rows: {blank_total_charges.sum()}")

df_clean.loc[blank_total_charges, ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head(11)


TotalCharges dtype before cleaning: object
Blank TotalCharges rows: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


## Clean TotalCharges

The blank `TotalCharges` rows belong to customers with `tenure == 0`, so replacing blanks with `0` is consistent with a newly started account that has not accumulated historical charges yet.


In [4]:
df_clean['TotalCharges'] = df_clean['TotalCharges'].replace(' ', pd.NA)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0.0)

print(f"TotalCharges dtype after cleaning: {df_clean['TotalCharges'].dtype}")
print(f"Missing TotalCharges after cleaning: {df_clean['TotalCharges'].isna().sum()}")
print(f"Rows with tenure == 0 and TotalCharges == 0: {((df_clean['tenure'] == 0) & (df_clean['TotalCharges'] == 0)).sum()}")


TotalCharges dtype after cleaning: float64
Missing TotalCharges after cleaning: 0
Rows with tenure == 0 and TotalCharges == 0: 11


## Verify Cleaned Dataset


In [5]:
display(df_clean.info())

missing_values = df_clean.isnull().sum()
missing_values = missing_values[missing_values > 0]

if missing_values.empty:
    print("No missing values found in the cleaned dataset.")
else:
    display(missing_values.sort_values(ascending=False))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


None

No missing values found in the cleaned dataset.


## Save Cleaned Data


In [6]:
output_path = project_root / 'data/interim/telco_churn_cleaned.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Saved shape: {df_clean.shape}")


Cleaned dataset saved to: /Users/mohammadmubashir/VCode/Churn/data/interim/telco_churn_cleaned.csv
Saved shape: (7043, 21)
